# Azzaro: Whisper turbo vs small vs large

**Objetivo:** comparar los tres modelos sobre el mismo video y escuchar los casos
donde producen transcripciones distintas.

Los clips se construyen con timestamps por palabra. El rango usado para cortar el
video es el mismo rango usado para recuperar el texto de cada modelo.


## 1. Configuracion


In [ ]:
from itertools import combinations
from pathlib import Path
import sys

try:
    ROOT = Path.cwd().resolve()
except FileNotFoundError:
    ROOT = Path.home() / "labios-argentos"

while not (ROOT / "requirements.txt").exists() and ROOT != ROOT.parent:
    ROOT = ROOT.parent

if not (ROOT / "requirements.txt").exists():
    raise FileNotFoundError(
        "No se encontro la raiz del repo. Abrir labios-argentos en VSCode y reiniciar el kernel."
    )

sys.path.insert(0, str(ROOT))

import pandas as pd
from IPython.display import Markdown, Video, display

from evaluation.src.whisper_model_comparison import (
    alinear_diferencias,
    construir_casos_tres_modelos,
    descargar_video_yt,
    exportar_clips_revision_webm,
    extraer_palabras,
    resumen_diferencias,
    transcribir_whisper,
)

VIDEO_URL = "https://www.youtube.com/watch?v=a4ggqJZXnQE"
VIDEO_PATH = None
MODELOS = ["turbo", "small", "large"]
COOKIES_FROM_BROWSER = "chrome"
FORCE_TRANSCRIBE = False
FORCE_CLIPS = True

# IDs recalculados con timestamps por palabra.
# Poner None para revisar todos los casos donde difieren los tres modelos.
CASOS_RELEVANTES = [3, 5, 6, 15, 16, 20, 32, 40, 43, 58, 59, 65]

WORKDIR = ROOT / "evaluation" / "outputs" / "azzaro_whisper"
WORKDIR


## 2. Video y transcripciones


In [ ]:
if VIDEO_PATH:
    video_path = Path(VIDEO_PATH).expanduser().resolve()
else:
    video_path = descargar_video_yt(
        VIDEO_URL,
        WORKDIR / "videos",
        nombre_base="azzaro_racing_caracas",
        cookies_from_browser=COOKIES_FROM_BROWSER,
    )

resultados = {}
palabras = {}

for modelo in MODELOS:
    print(f"Cargando o transcribiendo: {modelo}")
    resultados[modelo] = transcribir_whisper(
        video_path,
        model_name=modelo,
        output_dir=WORKDIR / "transcripts",
        force=FORCE_TRANSCRIBE,
    )
    palabras[modelo] = extraer_palabras(resultados[modelo], modelo)

display(pd.DataFrame([
    {"modelo": modelo, "palabras": len(palabras[modelo])}
    for modelo in MODELOS
]))


## 3. Contar diferencias


In [ ]:
resumenes = []
for modelo_a, modelo_b in combinations(MODELOS, 2):
    diferencias = alinear_diferencias(
        palabras[modelo_a],
        palabras[modelo_b],
        contexto=0,
    )
    resumen = resumen_diferencias(
        diferencias,
        palabras[modelo_a],
        palabras[modelo_b],
    )
    resumenes.append({
        "comparacion": f"{modelo_a} vs {modelo_b}",
        "grupos_distintos": resumen["grupos_con_diferencias"],
        f"palabras_{modelo_a}": resumen["palabras_turbo_en_diferencias"],
        f"palabras_{modelo_b}": resumen["palabras_small_en_diferencias"],
    })

display(pd.DataFrame(resumenes))

casos = construir_casos_tres_modelos(palabras)
if CASOS_RELEVANTES is None:
    casos_revision = casos
else:
    ids = set(CASOS_RELEVANTES)
    casos_revision = [caso for caso in casos if caso["diff_id"] in ids]

print(f"Casos donde los tres modelos difieren: {len(casos)}")
print(f"Casos seleccionados para revision: {len(casos_revision)}")

display(pd.DataFrame(casos_revision)[[
    "diff_id",
    "start",
    "end",
    "transcripcion_turbo",
    "transcripcion_small",
    "transcripcion_large",
]])


## 4. Casos mas relevantes

Cada reproductor usa WebM con VP8 y Vorbis para que VSCode pueda decodificar video
y audio. El texto de los tres modelos corresponde al mismo intervalo mostrado.


In [ ]:
clips = exportar_clips_revision_webm(
    video_path,
    casos_revision,
    WORKDIR / "clips_alineados_webm",
    margen=0.08,
    force=FORCE_CLIPS,
)

for numero, caso in enumerate(clips, start=1):
    display(Markdown(
        f"### Caso relevante {numero} | diff_{int(caso['diff_id']):03d} | "
        f"{caso['clip_start']}s-{caso['clip_end']}s"
    ))

    display(Video(
        filename=caso["clip_path"],
        embed=True,
        mimetype="video/webm",
        html_attributes="controls preload='metadata'",
    ))

    print("turbo:", caso["transcripcion_turbo"])
    print("small:", caso["transcripcion_small"])
    print("large:", caso["transcripcion_large"])
